# Auditer un serveur MCP qu'on n'a pas ecrit

*Classifier les outils d'un catalogue MCP en CRUD generique vs verbes metier,
et mesurer la distance de chaque outil au schema de persistance. Recycle la
documentation du projet LivresAgites (en sommeil) vers le depot pedagogique
[CoursIA](https://github.com/jsboige/CoursIA). Aucune donnee du site observe
n'est reproduite : le catalogue est **entierement synthetique** (Maison
Valmont).*

## La these

> **Un serveur MCP utile expose les verbes du metier, pas les tables de la
> base.**

Cette these est facile a proclamer et difficile a verifier. Un catalogue MCP
melange deux natures d'outils sans les etiqueter :

- du **CRUD generique** (creer / lire / mettre a jour / supprimer des rangees
  de tables), qui donne a l'agent un pouvoir maximal et une comprehension
  minimale ;
- des **verbes metier** (soumettre, assigner, faire passer d'un statut a un
  autre), qui restreignent le geste possible et y attachent le sens.

Les distinguer a l'oeil sur 90 outils est fastidieux et subjectif. Ce notebook
construit la **mesure** : etant donne le `inputSchema` d'un outil, on estime sa
*distance au schema de persistance* — un CRUD ressemble a une ligne de table,
un verbe metier non. La separation en deux clusters est le resultat cherché.


## Le catalogue de l'atelier : Maison Valmont

Maison Valmont est une **maison d'edition fictive** qui tourne sur un CMS
generique. Son serveur MCP expose deux couches d'outils :

| Couche | Prefixe | Nature | Exemples |
|--------|---------|--------|----------|
| Plateforme (CMS + boutique) | `cms_`, `store_` | CRUD sur les tables | `cms_create_post`, `store_list_orders` |
| Metier editorial | `valmont_` | Verbes du workflow | `valmont_submit_manuscript`, `valmont_assign_reviewer` |

La couche plateforme decrit le **CMS** ; la couche metier decrit
l'**edition**. Un agent qui n'aurait que la premiere devrait reconstituer la
logique editoriale a coups de requetes sur des tables — fragile et dangereux.

> **Note de methode.** Ce catalogue est synthetique. Aucun outil, aucun nom de
> domaine, aucune prose ne provient d'une installation reelle. Pour auditer
> **votre** serveur MCP (AI-Engine ou autre), renseigner `MCP_TOOL_CATALOG_URL`
> dans `.env` — voir le fichier `.env.example`. Sans `.env`, le notebook
> utilise le catalogue embarque ci-dessous.


In [1]:
#dependances : stdlib uniquement (json, re, os, base64, urllib). Aucun paquet externe.
# Aucun reseau n'est necessaire pour le catalogue de l'atelier ; l'instance
# jetable, si elle tourne en local, est interrogeee en direct.
import base64
import json, re, os
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError

# --- Chargement du catalogue --------------------------------------------
# Trois sources, dans cet ordre :
#   1. Un endpoint MCP declare dans l'environnement (MCP_TOOL_CATALOG_URL),
#      avec un jeton Bearer optionnel (MCP_TOOL_TOKEN).
#   2. Le miroir REST de l'instance jetable (variables VALMONT_*) :
#      GET /wp-json/mwai/v1/mcp/functions liste les memes outils avec leur
#      niveau d'acces (accessLevel). L'authentification est un mot de passe
#      d'application, lu dans instance-jetable/.env.
#   3. Le catalogue synthetique Maison Valmont embarque (defaut hors instance).
#
# Les identifiants ne sont jamais imprimes : on dit seulement s'ils ont ete
# trouves, et depuis quel fichier.

CANDIDATS_ENV = (
    "instance-jetable/.env",
    "../../instance-jetable/.env",
    "../instance-jetable/.env",
    "AI-Engine-WordPress/instance-jetable/.env",
)


def lire_env(*candidats):
    """Lit un .env minimal (CLE=valeur), sans dependance externe."""
    for candidat in candidats:
        p = Path(candidat)
        if not p.is_file():
            continue
        env = {}
        for ligne in p.read_text(encoding="utf-8").splitlines():
            ligne = ligne.strip()
            if not ligne or ligne.startswith("#") or "=" not in ligne:
                continue
            cle, _, valeur = ligne.partition("=")
            env[cle.strip()] = valeur.strip().strip('"').strip("'")
        return env, str(p)
    return {}, None


ENV_LOCAL, ENV_PATH = lire_env(*CANDIDATS_ENV)
print("Fichier d'environnement de l'instance :", ENV_PATH or "(aucun)")
print("Identifiants d'application trouves  :",
      bool(ENV_LOCAL.get("VALMONT_ADMIN_USER")) and bool(ENV_LOCAL.get("VALMONT_APP_PASSWORD")))

def charger_catalogue():
    """Charge le catalogue : endpoint MCP, puis instance, puis catalogue embarque."""
    url = os.environ.get("MCP_TOOL_CATALOG_URL", "").strip()
    if url:
        token = os.environ.get("MCP_TOOL_TOKEN", "").strip()
        headers = {"Accept": "application/json"}
        if token:
            headers["Authorization"] = f"Bearer {token}"
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=10) as r:
                data = json.loads(r.read().decode("utf-8"))
            if isinstance(data, dict):
                data = data.get("functions", data.get("tools", []))
            print(f"Catalogue charge depuis {url} : {len(data)} outils.")
            return normaliser(data)
        except (URLError, ValueError, OSError) as e:
            print(f"Endpoint live injoignable ({e}). Bascule sur l'instance ou le catalogue embarque.")
    instance = charger_miroir_instance()
    if instance is not None:
        return instance
    print(f"Catalogue synthetique Maison Valmont : {len(CATALOGUE_VALMONT)} outils.")
    return normaliser(CATALOGUE_VALMONT)

def charger_miroir_instance():
    """Catalogue reellement expose par l'instance jetable (miroir REST du plugin).

    Retourne None si l'instance n'est pas configuree ou pas joignable :
    l'appelant bascule alors sur le catalogue embarque.
    """
    base = (ENV_LOCAL.get("VALMONT_BASE_URL") or "").rstrip("/")
    user = ENV_LOCAL.get("VALMONT_ADMIN_USER", "")
    pwd = ENV_LOCAL.get("VALMONT_APP_PASSWORD", "")
    if not (base and user and pwd):
        print("Instance locale : configuration absente (base, utilisateur, mot de passe).")
        return None
    url = base + "/wp-json/mwai/v1/mcp/functions"
    creds = base64.b64encode(f"{user}:{pwd}".encode()).decode()
    req = Request(url, headers={"Authorization": "Basic " + creds,
                                "Accept": "application/json"})
    try:
        with urlopen(req, timeout=15) as r:
            data = json.loads(r.read().decode("utf-8"))
    except (URLError, ValueError, OSError) as e:
        print(f"Instance locale injoignable ({e}).")
        return None
    fonctions = data.get("functions", []) if isinstance(data, dict) else data
    print(f"Catalogue expose par l'instance : {len(fonctions)} outils ({url}).")
    return normaliser(fonctions)

def normaliser(outils_bruts):
    """Unifie le format : {name, description, params (dict name->type), accessLevel}."""
    out = []
    for o in outils_bruts:
        schema = o.get("inputSchema", {}) or {}
        props = schema.get("properties", {}) or {}
        params = {p: meta.get("type", "?") for p, meta in props.items()}
        out.append({
            "name": o.get("name", "?"),
            "description": o.get("description", ""),
            "params": params,
            "accessLevel": o.get("accessLevel", ""),
        })
    return out

CATALOGUE_VALMONT = [
    # --- Couche plateforme : CRUD generique (CMS + boutique) ---
    {"name": "cms_create_post", "description": "Creer un article.",
     "inputSchema": {"type": "object", "properties": {
        "title": {"type": "string"}, "content": {"type": "string"},
        "status": {"type": "string"}, "author_id": {"type": "integer"},
        "slug": {"type": "string"}, "excerpt": {"type": "string"},
        "date": {"type": "string"}, "categories": {"type": "array"},
        "tags": {"type": "array"}}}},
    {"name": "cms_update_post", "description": "Modifier un article.",
     "inputSchema": {"type": "object", "properties": {
        "post_id": {"type": "integer"}, "title": {"type": "string"},
        "content": {"type": "string"}, "status": {"type": "string"},
        "slug": {"type": "string"}}}},
    {"name": "cms_get_post", "description": "Lire un article.",
     "inputSchema": {"type": "object", "properties": {"post_id": {"type": "integer"}}}},
    {"name": "cms_list_posts", "description": "Lister les articles.",
     "inputSchema": {"type": "object", "properties": {
        "status": {"type": "string"}, "author_id": {"type": "integer"},
        "limit": {"type": "integer"}, "offset": {"type": "integer"},
        "order": {"type": "string"}}}},
    {"name": "cms_delete_post", "description": "Supprimer un article.",
     "inputSchema": {"type": "object", "properties": {
        "post_id": {"type": "integer"}, "force": {"type": "boolean"}}}},
    {"name": "cms_list_users", "description": "Lister les comptes.",
     "inputSchema": {"type": "object", "properties": {
        "role": {"type": "string"}, "limit": {"type": "integer"},
        "offset": {"type": "integer"}}}},
    {"name": "cms_get_user", "description": "Lire un compte.",
     "inputSchema": {"type": "object", "properties": {"user_id": {"type": "integer"}}}},
    {"name": "cms_list_media", "description": "Lister les medias.",
     "inputSchema": {"type": "object", "properties": {
        "limit": {"type": "integer"}, "offset": {"type": "integer"},
        "mime_type": {"type": "string"}}}},
    {"name": "store_create_product", "description": "Creer un produit.",
     "inputSchema": {"type": "object", "properties": {
        "name": {"type": "string"}, "sku": {"type": "string"},
        "price": {"type": "number"}, "stock": {"type": "integer"},
        "description": {"type": "string"}, "categories": {"type": "array"},
        "weight": {"type": "number"}}}},
    {"name": "store_update_product", "description": "Modifier un produit.",
     "inputSchema": {"type": "object", "properties": {
        "product_id": {"type": "integer"}, "name": {"type": "string"},
        "price": {"type": "number"}, "stock": {"type": "integer"}}}},
    {"name": "store_list_orders", "description": "Lister les commandes.",
     "inputSchema": {"type": "object", "properties": {
        "status": {"type": "string"}, "customer_id": {"type": "integer"},
        "limit": {"type": "integer"}, "offset": {"type": "integer"}}}},
    {"name": "store_get_order", "description": "Lire une commande.",
     "inputSchema": {"type": "object", "properties": {"order_id": {"type": "integer"}}}},
    # --- Couche metier : verbes du workflow editorial ---
    {"name": "valmont_submit_manuscript", "description": "Deposer un manuscrit.",
     "inputSchema": {"type": "object", "properties": {
        "author_id": {"type": "integer"}, "title": {"type": "string"},
        "genre": {"type": "string"}, "synopsis": {"type": "string"},
        "manuscript_file": {"type": "string"}}}},
    {"name": "valmont_assign_reviewer", "description": "Assigner un relecteur.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"}, "reviewer_id": {"type": "integer"}}}},
    {"name": "valmont_submit_reading_report", "description": "Deposer un rapport de lecture.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"}, "reviewer_id": {"type": "integer"},
        "decision": {"type": "string", "enum": ["favorable", "reserve", "defavorable"]},
        "comment": {"type": "string"}}}},
    {"name": "valmont_transition_manuscript", "description": "Faire passer un manuscrit de statut.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"}, "from_status": {"type": "string"},
        "to_status": {"type": "string"}}}},
    {"name": "valmont_get_pending_decisions", "description": "Decisions en attente du comite.",
     "inputSchema": {"type": "object", "properties": {"committee_id": {"type": "integer"}}}},
    {"name": "valmont_get_manuscripts", "description": "Lister les manuscrits.",
     "inputSchema": {"type": "object", "properties": {
        "status": {"type": "string"}, "author_id": {"type": "integer"},
        "limit": {"type": "integer"}, "offset": {"type": "integer"}}}},
    {"name": "valmont_request_prereading", "description": "Demander une prelecture assistee.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"}, "environment": {"type": "string"}}}},
    {"name": "valmont_notify_author", "description": "Notifier l'autrice d'un evenement.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"}, "event": {"type": "string"}}}},
    {"name": "valmont_query_catalog", "description": "Interroger le catalogue indexe.",
     "inputSchema": {"type": "object", "properties": {
        "query": {"type": "string"}, "environment": {"type": "string"},
        "k": {"type": "integer"}}}},
    {"name": "valmont_get_processing_status", "description": "Etat du pipeline d'un manuscrit.",
     "inputSchema": {"type": "object", "properties": {
        "manuscript_id": {"type": "integer"},
        "pipeline_stage": {"type": "string",
            "enum": ["extraction", "chunking", "indexing", "prereading"]}}}},
]

OUTILS = charger_catalogue()
print(f"Total charge : {len(OUTILS)} outils.")
OUTILS_ATELIER = normaliser(CATALOGUE_VALMONT)
print(f"Catalogue de l'atelier (reference, toujours disponible) : {len(OUTILS_ATELIER)} outils.")


Fichier d'environnement de l'instance : instance-jetable\.env
Identifiants d'application trouves  : True
Catalogue expose par l'instance : 42 outils (http://localhost:8093/wp-json/mwai/v1/mcp/functions).
Total charge : 42 outils.
Catalogue de l'atelier (reference, toujours disponible) : 22 outils.


## Premiere lecture : compter par prefixe

Le reflexe est de classer par prefixe. Donnons-lui raison sur les comptes,
puis montrons sa limite.

Le prefixe est un **indice**, pas une **definition**. Un outil prefinance
`valmont_` peut tres bien etre un CRUD degenere ; un outil prefinance `cms_`
peut encapsuler une regle. Verifions.


In [2]:
def prefixe(nom):
    return nom.split("_", 1)[0] + "_"

comptes = Counter(prefixe(o["name"]) for o in OUTILS)
print("Outils par prefixe :")
for p, n in sorted(comptes.items(), key=lambda kv: -kv[1]):
    print(f"  {p:<12} {n}")

# Prefixes de plateforme : les CRUD generiques d'une plateforme (CMS,
# boutique, WordPress). Tout autre prefixe annonce un espace de noms propre
# -- que l'instrument de la cellule suivante confronte au schema.
PREFIXES_PLATEFORME = {"cms_", "store_", "wp_"}
n_plateforme = sum(n for p, n in comptes.items() if p in PREFIXES_PLATEFORME)
n_hors = len(OUTILS) - n_plateforme
print(f"\nCRUD via prefixe (plateforme) : {n_plateforme} / {len(OUTILS)}")
print("Hors prefixe plateforme      :", n_hors)


Outils par prefixe :
  wp_          40
  mwai_        2

CRUD via prefixe (plateforme) : 40 / 42
Hors prefixe plateforme      : 2


## La vraie question : l'outil parle-t-il la base ou le metier ?

Compter par prefixe ne dit rien de l'**interieur** d'un outil. Ce qui distingue
un CRUD d'un verbe metier, c'est la **forme de son schema** :

- un **CRUD** expose les **colonnes** d'une table (beaucoup de champs
  libres : `title`, `content`, `price`, `slug`...). L'agent ecrit la ligne.
- un **verbe metier** encapsule une regle et n'expose que ce que l'appelant
  doit decider (`manuscript_id`, `decision`, `from_status`/`to_status`). La
  regle vit dans l'outil, pas chez l'agent.

On mesure donc trois signaux tires du `inputSchema` :

1. **Presence de mots-clés workflow** (`decision`, `reviewer_id`,
   `from_status`, `to_status`, `environment`, `pipeline_stage`...) : un CRUD
   n'en a aucun ; un verbe metier en a presque toujours.
2. **Largeur du schema** : un CRUD create/update est large (5-9 champs
   inscriptibles) ; un verbe metier est etroit (2-4) parce qu'il cache la table.
3. **Identifiant d'entite generique** (`post_id`, `product_id`, `order_id`...) :
   signal CRUD ; un verbe metier reference des concepts metier, pas des row ids.

On combine ces signaux en un score de **distance au schema de persistance**,
entre 0 (tout-CRUD) et 1 (tout-metier).

Un quatrieme signal complete la lecture, et il est **structurel** : le **nom**
de l'outil annonce un objet du metier (un manuscrit, une image), et le schema
dit si l'appel sait nommer cet objet. Quand le nom promet et que le schema se
tait, l'outil est un **imposteur** -- il porte une promesse metier qu'il ne
sait pas tenir.


In [3]:
# --- Vocabulaires de reference -----------------------------------------
# Parametres qui sentent la colonne de table (CRUD).
COLONNES = {
    "title", "content", "status", "slug", "excerpt", "date", "categories",
    "tags", "role", "limit", "offset", "order", "force", "mime_type",
    "name", "sku", "price", "stock", "description", "weight", "email",
    # identifiants de row generiques (un CRUD pointe une rangee).
    "post_id", "user_id", "product_id", "order_id", "comment_id", "media_id",
    "customer_id", "term_id",
    # --- Idiom WordPress : le vocabulaire d'un catalogue d'instance ------
    "ID", "fields", "field", "flags", "regex", "replace", "key", "value",
    "meta", "post_title", "post_content", "post_excerpt", "post_name",
    "post_status", "post_type", "meta_input", "taxonomy", "term_name",
    "comment_ID", "user_email", "user_login", "display_name", "author",
    "author_name", "author_email", "author__not_in", "paged", "search",
    "after", "before", "parent", "append", "terms", "raw", "comment_content",
    "comment_author", "comment_author_email", "comment_author_url",
    "comment_approved", "alt", "caption", "url", "path", "filename",
    "base64", "mode", "blocks", "pattern", "type", "include", "exclude",
    "content_format",
}
# Parametres qui sentent le workflow (verbe metier).
WORKFLOW = {
    "decision", "reviewer_id", "from_status", "to_status", "committee_id",
    "environment", "event", "pipeline_stage", "genre", "synopsis",
    "manuscript_file", "query", "k", "manuscript_id", "author_id",
    # Une publication planifiee est un workflow, meme si elle s'ecrit dans
    # la meme table que les autres.
    "schedule_for",
}
VERBES_CRUD = {"create", "update", "delete", "get", "list", "set", "edit",
               "add", "remove", "fetch"}
VERBES_METIER = {"submit", "assign", "approve", "reject", "request",
                 "transition", "notify", "query", "send", "schedule"}

# --- Les objets du metier ------------------------------------------------
# Ce que le NOM d'un outil promet de manipuler. Un objet du metier est une
# entite de la maison (un manuscrit, une image, un comite) -- pas une notion
# generique de la plateforme. Le NOM est une declaration ; le schema, un fait.
OBJETS_METIER = {
    "manuscript", "manuscrit", "committee", "comite", "reviewer",
    "relecteur", "decision", "prereading", "prelecture", "pipeline",
    "image", "vision", "edition", "chapter", "chapitre", "proof", "isbn",
    "rights", "droits", "royalty", "translation", "traduction", "synopsis",
    "editorial", "reading", "lecture",
}
# Entites que l'appel sait nommer : un parametre qui adresse l'objet.
ENTITES_SCHEMA = {
    "manuscript_id", "manuscript_file", "committee_id", "reviewer_id",
    "decision", "pipeline_stage", "media_id",
}

def verbe(nom):
    return nom.split("_", 1)[1].split("_")[0].lower() if "_" in nom else ""

def signaux(outil):
    params = set(outil["params"])
    n = max(len(params), 1)
    v = verbe(outil["name"])
    return {
        "n": len(params),
        "verbe": v,
        "verb_crud": v in VERBES_CRUD,
        "verb_metier": v in VERBES_METIER,
        "kw_workflow": len(params & WORKFLOW),
        "large": len(params) >= 5,
        "id_entite": len(params & {"post_id", "user_id", "product_id",
                                   "order_id", "comment_id", "media_id",
                                   "customer_id", "ID", "comment_ID"}),
        "overlap_colonnes": len(params & COLONNES) / n,
    }

def distance(outil):
    """Score [0,1]. Haut = metier (loin de la table) ; bas = CRUD."""
    s = signaux(outil)
    score = 0.0
    if s["verb_metier"]: score += 0.30
    if s["verb_crud"]:   score -= 0.30
    score += min(s["kw_workflow"], 2) * 0.20      # chaque kw workflow pousse vers le haut
    if s["id_entite"]:   score -= 0.20
    score += (1.0 - s["overlap_colonnes"]) * 0.25 # moins de colonnes -> plus loin
    if s["large"]:       score -= 0.10            # un schema large est une row a ecrire
    # mapage approximatif [-0.65, +0.75] -> [0, 1]
    d = (score + 0.65) / 1.40
    return max(0.0, min(1.0, d))

def analyser(catalogue):
    """Applique l'instrument a un catalogue entier."""
    ann = []
    for o in catalogue:
        d = distance(o)
        objets = sorted({w for w in OBJETS_METIER if w in o["name"].lower()})
        entites = sorted(set(o["params"]) & ENTITES_SCHEMA)
        ann.append({**o, "distance": d,
                    "classe": "metier" if d >= 0.5 else "CRUD",
                    "objets": objets, "entites": entites,
                    # Un imposteur annonce un objet du metier que l'appel ne
                    # sait pas nommer : la promesse est au nom, le schema ne
                    # la soutient pas.
                    "imposteur": bool(objets) and not entites})
    return ann

# Annotons chaque outil.
ANALYSE = analyser(OUTILS)
print("Classification calculee.")


Classification calculee.


## Le resultat : ce que le score revele

Classons tous les outils par distance croissante et regardons la distribution.
Si la these tient, le score separe le catalogue en deux groupes nets -- avec,
eventuellement, un cas limite interessant. Ce que le score revele depend du
catalogue audite : deux groupes sur le catalogue de l'atelier, un seul sur le
catalogue expose par l'instance.


In [4]:
def barre(d, largeur=30):
    n = int(round(d * largeur))
    return "#" * n + "." * (largeur - n)

print(f"{'outil':<30} {'dist.':>6}  {'classe':<7} {'imp.':<4}  courbe")
print("-" * 86)
for a in sorted(ANALYSE, key=lambda x: x["distance"]):
    print(f"{a['name']:<30} {a['distance']:>6.2f}  {a['classe']:<7} "
          f"{'OUI' if a['imposteur'] else '-':<4}  {barre(a['distance'])}")

n_crud = sum(1 for a in ANALYSE if a["classe"] == "CRUD")
n_metier = sum(1 for a in ANALYSE if a["classe"] == "metier")
n_imposteurs = sum(1 for a in ANALYSE if a["imposteur"])
n_verifies = n_metier - sum(1 for a in ANALYSE if a["imposteur"] and a["classe"] == "metier")
print(f"\nTotal : {n_crud} CRUD, {n_metier} metier, dont {n_verifies} verifies.")
print(f"Imposteurs (nom promis, schema muet) : {n_imposteurs}")
for a in sorted(ANALYSE, key=lambda x: x["distance"]):
    if a["imposteur"]:
        print(f"  {a['name']:<26} annonce {a['objets']} ; parametres : {sorted(a['params'])}")


outil                           dist.  classe  imp.  courbe
--------------------------------------------------------------------------------------
wp_get_comments                  0.04  CRUD    -     #.............................
wp_create_comment                0.04  CRUD    -     #.............................
wp_update_media                  0.04  CRUD    -     #.............................
wp_update_user                   0.11  CRUD    -     ###...........................
wp_update_comment                0.11  CRUD    -     ###...........................
wp_delete_comment                0.11  CRUD    -     ###...........................
wp_get_post                      0.11  CRUD    -     ###...........................
wp_get_post_snapshot             0.11  CRUD    -     ###...........................
wp_delete_post                   0.11  CRUD    -     ###...........................
wp_get_post_meta                 0.11  CRUD    -     ###...........................
wp_update_pos

In [5]:
# --- Deux catalogues, deux audits ---------------------------------------
# L'instance expose un catalogue ; l'atelier en a un de reference. Le meme
# instrument s'applique aux deux, et le catalogue expose est publie en
# detail : le nom de chaque outil et son niveau d'acces.

AUDIT_ATELIER = analyser(OUTILS_ATELIER)

def resume(audit):
    return {
        "outils": len(audit),
        "CRUD": sum(1 for a in audit if a["classe"] == "CRUD"),
        "metier": sum(1 for a in audit if a["classe"] == "metier"),
        "metier_verifie": sum(1 for a in audit
                              if a["classe"] == "metier" and not a["imposteur"]),
        "imposteurs": sum(1 for a in audit if a["imposteur"]),
    }

R_EXPOSE = resume(ANALYSE)
R_ATELIER = resume(AUDIT_ATELIER)
print("Catalogue expose par l'instance :", R_EXPOSE)
print("Catalogue de l'atelier           :", R_ATELIER)
print()
print("Niveaux d'acces du catalogue expose :")
niveaux = {}
for a in ANALYSE:
    niveaux[a["accessLevel"] or "(non declare)"] = niveaux.get(a["accessLevel"] or "(non declare)", 0) + 1
for niveau, n in sorted(niveaux.items()):
    print(f"  {niveau:<15} {n}")
print()
print("Catalogue expose, outil par outil (nom | niveau d'acces) :")
for a in sorted(ANALYSE, key=lambda x: (x["accessLevel"], x["name"])):
    print(f"  {a['name']:<26} {a['accessLevel'] or '(non declare)'}")


Catalogue expose par l'instance : {'outils': 42, 'CRUD': 41, 'metier': 1, 'metier_verifie': 0, 'imposteurs': 2}
Catalogue de l'atelier           : {'outils': 22, 'CRUD': 13, 'metier': 9, 'metier_verifie': 9, 'imposteurs': 1}

Niveaux d'acces du catalogue expose :
  admin           10
  read            16
  write           16

Catalogue expose, outil par outil (nom | niveau d'acces) :
  wp_create_user             admin
  wp_delete_comment          admin
  wp_delete_media            admin
  wp_delete_post             admin
  wp_delete_post_meta        admin
  wp_delete_term             admin
  wp_get_option              admin
  wp_get_users               admin
  wp_update_option           admin
  wp_update_user             admin
  mwai_vision                read
  wp_count_media             read
  wp_count_posts             read
  wp_count_terms             read
  wp_get_comments            read
  wp_get_media               read
  wp_get_post                read
  wp_get_post_meta       

### Lecture

Sur le catalogue reellement expose, la lecture est nette : **42 outils, dont 41
classes CRUD**. Le seul que le score range du cote metier est `mwai_vision`
(distance 0,52) -- et c'est un **imposteur**.

Un imposteur est un outil dont le nom annonce un objet du metier que l'appel
ne sait pas nommer. Ici, le nom promet une image, et le schema ne dit pas
comment la nommer :

- `mwai_vision` (distance 0,52) : le nom promet une **vision**, les parametres
  sont `message`, `path`, `url` -- l'appel ne nomme jamais l'image ;
- `mwai_image` (distance 0,45) : le nom promet une **image**, les parametres
  sont `alt`, `caption`, `description`, `message`, `postId`, `title` -- les
  metadonnees d'une ligne de media, pas une image.

Le compte est donc : **2 imposteurs, 0 verbe metier verifie**. Les 41 autres
outils sont des operations de table -- creer, lire, modifier, supprimer des
lignes de `wp_posts`, `wp_users`, `wp_comments`, `wp_options`, `wp_terms` ou
`wp_media`. Aucun ne dit quoi faire d'un manuscrit.

Sur le catalogue de l'atelier, le meme instrument donne l'autre extreme :
**22 outils, 13 CRUD, 9 metier verifies** -- et un imposteur,
`valmont_get_manuscripts`, dont le prefixe annonce les manuscrits alors que
son schema (`status`, `author_id`, `limit`, `offset`) est la signature exacte
d'une liste filtree sur une table. Le prefixe ment ; le schema, non.

C'est la lecon operationnelle :

> **Le prefixe d'un outil est une declaration d'intention. Sa distance au
schema de persistance est un fait.** Quand un agent choisit mal, c'est
presque toujours parce qu'il a suivi le prefixe et non la forme.

Et la pire situation n'est pas l'outil mal classe -- c'est l'**absence** de
verbe metier la ou une regle devrait vivre. Sur l'instance, ce manque se
chiffre : pour qu'un agent fasse respecter une regle du type « *un manuscrit
ne passe en relecture qu'avec deux relecteurs assignes* », aucun outil n'existe
ou l'ecrire ; il ne reste qu'une ecriture de table, et rien ne l'empechera de
sauter un statut interdit.


## La lecon, mesurée

La these de depart (« un serveur MCP utile expose les verbes du metier »)
n'etait qu'une affirmation tant qu'elle reposait sur l'intuition. Mesurer la
distance au schema de persistance en fait une **propriete observable** du
catalogue :

- elle est **reproductible** : deux auditeurs trouvent le meme classement sur
  le meme catalogue ;
- elle est **actionnable** : sur le catalogue expose, 41 outils sur 42 sont
  classes CRUD et aucun n'est un verbe metier verifie -- la regle metier
  n'est encapsulee nulle part, et l'agent est livre a lui-meme sur les
  transitions interdites ;
- elle est **portable** : la methode s'applique a n'importe quel serveur MCP,
  n'importe quel CMS, n'importe quel langage -- il suffit du `inputSchema`.

Le contre-pied vaut aussi : un catalogue ou *tout* serait un verbe metier
(score > 0,8 partout) serait tout aussi suspect -- il manquerait le CRUD
elementaire dont un agent a besoin pour lire l'etat du monde. L'audit sain
cherche un **mix conscient**, pas un extreme.

Ce que l'audit ajoute ici n'est pas une these, c'est un **ecart** mesure
entre deux catalogues de la meme famille. Sur l'instance gratuite, la lecon
est deja ecrite dans le carnet voisin
([`piloter-wordpress-par-mcp.ipynb`](piloter-wordpress-par-mcp.ipynb),
section « Ce qu'on en a fait dans le projet Livres Agites ») : en version
payante, le serveur expose 88 outils dont 24 metier ecrits sur mesure ;
en version gratuite, les 43 outils de `tools/list` sont tous generiques. Le
present audit mesure le meme ecart par une autre voie -- sur le miroir REST :
42 outils exposes, 41 classes CRUD, 0 verbe metier verifie, et les deux
seules promesses du metier (l'image a generer, l'image a analyser) portees
par des imposteurs.


## Exercices

Les exercices s'exercent sur le **catalogue de l'atelier**, qui reste embarque
dans le carnet : ils sont donc faisables hors reseau, sur n'importe quelle
machine. Le catalogue de l'instance, quand elle est joignable, s'analyse avec
le meme instrument -- c'est l'objet de la lecture qui precede.


### Exercice 1 -- Classer sans le prefixe

Le prefixe est un indice trop commode. Ecrire `classe_sans_prefixe(outil)` qui
renvoie `"CRUD"` ou `"metier"` en ignorant **totalement** le nom de l'outil --
en se fondant uniquement sur `outil["params"]`. Verifier qu'elle range
`valmont_get_manuscripts` du cote CRUD (catalogue de l'atelier).


In [6]:
# Exercice 1 -- classifier un outil SANS regarder son nom.
def classe_sans_prefixe(outil):
    params = set(outil["params"])
    # TODO: decider CRUD ou metier a partir des parametres seuls.
    # Indice : la presence d'un mot-cle workflow est le signal le plus fort.
    return None  # <- remplacer

# Test rapide (doit rendre 'CRUD' pour l'imposteur et 'metier' pour un verbe).
# imposteur = prochain(o for o in OUTILS_ATELIER if o["name"] == "valmont_get_manuscripts")
# verbe = prochain(o for o in OUTILS_ATELIER if o["name"] == "valmont_assign_reviewer")
# print(classe_sans_prefixe(imposteur), classe_sans_prefixe(verbe))


### Exercice 2 -- Trouver l'imposteur

Un imposteur annonce un objet du metier que son schema ne sait pas nommer.
Sur le catalogue de l'atelier, `valmont_get_manuscripts` en est un : le prefixe
annonce les manuscrits, le schema (`status`, `author_id`, `limit`, `offset`)
ne permet de nommer aucun manuscrit.

Ecrire `trouver_imposteur(catalogue)` qui renvoie la **liste des noms**
d'imposteurs d'un catalogue -- sans coder le nom en dur, en croisant les
objets annonces par le nom et les entites que le schema sait nommer.


In [7]:
# Exercice 2 -- identifier l'outil metier au schema CRUD.
def trouver_imposteur(catalogue):
    # TODO: renvoyer la liste des noms d'outils dont le nom annonce un objet
    # du metier (OBJETS_METIER) mais dont le schema ne nomme aucune entite
    # (ENTITES_SCHEMA).
    # Indice : analyser(catalogue) renvoie deja le drapeau "imposteur".
    return None  # <- remplacer


### Exercice 3 -- Concevoir un verbe metier

Le CRUD ne peut pas tout. Soit la regle :

> *Un manuscrit ne peut passer a l'etat `en_relecture` que s'il a au moins
> deux relecteurs assignes.*

Cette regle **doit** vivre dans un outil metier, pas chez l'agent. Ecrire
`concevoir_ouvrir_relecture()` qui renvoie un dict `{"name", "inputSchema"}`
encodant ce verbe. L'agent ne doit pouvoir fournir que `manuscript_id` ; la
verification des deux relecteurs est l'affaire de l'outil.


In [8]:
# Exercice 3 -- designer un verbe metier pour une regle de transition.
def concevoir_ouvrir_relecture():
    return {
        "name": "",  # <- nom du verbe
        "inputSchema": {"type": "object", "properties": {
            # TODO: les parametres que l'agent PEUT fournir.
        }},
    }


### Exercice 4 -- Auditer la derive d'un catalogue

Une refonte de serveur MCP est un moment dangereux : des outils disparaissent, d'autres apparaissent, d'autres sont renommes — et l'agent qui comptait sur l'ancien nom casse silencieusement. Le travail de l'auditeur ne s'arrete pas au catalogue figé : il produit le **rapport de derive** entre deux versions — `ajoutes`, `retires`, et les candidats `renomes` (un retire et un ajoute qui partagent leurs parametres).

**Indice de validation** : dans le jeu d'essai commente ci-dessous, `valmont_get_manuscripts` doit ressortir comme un renommage, pas comme un retrait suivi d'un ajout sans lien.


In [9]:
# Exercice 4 -- produire le rapport de derive entre deux versions du catalogue.
def diff_catalogue(v1, v2):
    # v1, v2 : listes d'outils [{"name": ..., "params": [...]}, ...]
    # TODO: renvoyer {"ajoutes": [...], "retires": [...], "renomes": [(ancien, nouveau), ...]}
    # Indice : ajoutes / retires = difference ensembliste sur les noms.
    # Indice : un retire et un ajoute qui partagent leurs parametres = candidat renomme.
    return None  # <- remplacer

# Test rapide (jeu d'essai factice, a decommenter) :
# V1 = [{"name": "valmont_get_manuscripts", "params": ["id"]},
#       {"name": "valmont_list_authors", "params": []}]
# V2 = [{"name": "valmont_fetch_manuscripts", "params": ["id"]},
#       {"name": "valmont_list_authors", "params": []},
#       {"name": "valmont_assign_reviewer", "params": ["id", "reviewer"]}]
# r = diff_catalogue(V1, V2)
# print(r["ajoutes"], r["retires"], r["renomes"])
# attendu : ['valmont_assign_reviewer'] | [] | [('valmont_get_manuscripts', 'valmont_fetch_manuscripts')]


## Provenance et pour aller plus loin

**Provenance.** Deux catalogues sont analyses par le meme instrument :

- le **catalogue de l'atelier** -- entierement synthetique (Maison Valmont),
  embarque dans le carnet : aucune donnee d'installation reelle, ni nom
  d'outil reel, ni titre d'ouvrage, ni prose sous droits, ni identifiant ;
- le **catalogue reellement expose** par l'instance jetable locale (AI Engine
  gratuit, plugin `mwai`) : 42 outils lus en direct sur
  `/wp-json/mwai/v1/mcp/functions`, avec leur niveau d'acces. Ce sont des
  noms d'outils et des niveaux d'acces -- **aucune donnee client, aucun
  secret** : le mot de passe d'application n'est jamais imprime, le carnet
  ne dit que s'il a ete trouve.

Hors instance, le catalogue de l'atelier prend le relais et le carnet
s'execute integralement : les chiffres changent, la methode ne bouge pas.

**Ce que ce notebook ne dit pas.** La distance au schema est une *heuristic*
lue sur le `inputSchema`. Elle ne saisit pas une regle encodee dans le **corps**
de l'outil (un CRUD pourrait cacher une regle cote serveur ; un verbe metier
pourrait n'etre qu'une coquille vide). L'audit par le schema est un **premier
filtre** ; il trouve les catalogues qui manquent de verbes metier, il ne prouve
pas que les verbes presents sont correctement implementes. Le niveau d'acces
lui-meme (`accessLevel`) est **declare** par le serveur : l'audit le publie, il
ne le verifie pas.

**Pour aller plus loin.**

- [`piloter-wordpress-par-mcp.ipynb`](piloter-wordpress-par-mcp.ipynb) -- le
  meme serveur vu par le protocole (handshake, `tools/list`, appels reels), et
  la lecon « zero outil metier en gratuit » : 88 outils dont 24 metier dans la
  version payante, 43 outils tous generiques en version gratuite (le miroir
  REST en compte 42 : l'ecart est `mcp_ping`). L'audit ci-dessus mesure le
  meme ecart par une autre voie.
- [`livresagites-parcours.md`](../../04-Cas-Usage-livresagites/livresagites-parcours.md) -- le cas d'usage
  original (Parcours 3) dont ce notebook est le compagnon executable.
- [`ingestion-corpus-long-rag.ipynb`](../03-3-RAG-et-Embeddings/ingestion-corpus-long-rag.ipynb) -- un
  autre recyclage du meme projet : la qualite du RAG se joue au chunking.
- [`cadrer-les-agents.md`](../../../cadrer-les-agents.md) -- l'autre moitie : ce
  qu'un assistant a le droit de faire (autorisations), en regard de qui il est.
- [Model Context Protocol](https://modelcontextprotocol.io/) -- la spec.
